# Fig 4 — E4 cache-availability boundary

Two stacked subplots sharing X (filter delay).
(a) Stacked bar: % of runs landing Ready vs ClampedToOldest.
(b) Bar+error: force-switch PTS gap on Ready cells.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "figures"))

import matplotlib.pyplot as plt
import numpy as np

from _data import e4_decision_counts, load_aggregate
from _style import (
    COL_CLAMPED,
    COL_READY,
    COLUMN_WIDTH_IN,
    apply_acm_style,
)

apply_acm_style()

CACHE_SIZE = 20

In [ ]:
counts = e4_decision_counts()  # one row per cell, sorted by delay_s
agg = load_aggregate("e4")

# Per-cell mean ± stdev of force_switch_pts_gap_ms, Ready cells only.
ready_only = agg[agg["relay_decision"] == "Ready"]
gap_by_delay = (
    ready_only.groupby(ready_only["cell_id"].str.extract(r"delay(\d+)").astype(int)[0])
    ["force_switch_pts_gap_ms"]
    .agg(["mean", "std", "count"])
    .reindex(counts["delay_s"])  # align order
)

In [ ]:
fig, (ax_top, ax_bot) = plt.subplots(
    2, 1, figsize=(COLUMN_WIDTH_IN, 3.0),
    sharex=True, constrained_layout=True,
    gridspec_kw={"height_ratios": [1, 1]},
)

x = np.arange(len(counts))

# (a) Stacked 100% bars.
ready_pct = counts["ready"] / counts["n_runs"] * 100
clamped_pct = counts["clamped"] / counts["n_runs"] * 100
ax_top.bar(x, ready_pct, color=COL_READY, label="Ready", width=0.7)
ax_top.bar(x, clamped_pct, bottom=ready_pct, color=COL_CLAMPED,
           label="ClampedToOldest", width=0.7)
ax_top.set_ylim(0, 100)
ax_top.set_ylabel("% of runs")
ax_top.set_title("(a) Relay decision distribution", loc="left", fontsize=8, pad=2)
ax_top.legend(loc="center right", frameon=False, fontsize=7)

# (b) PTS gap (Ready only).
means = gap_by_delay["mean"].fillna(0).values
stds = gap_by_delay["std"].fillna(0).values
ax_bot.bar(x, means, yerr=stds, color=COL_READY, alpha=0.85,
           width=0.7, capsize=3, error_kw={"elinewidth": 0.8})
for i, c in enumerate(gap_by_delay["count"].fillna(0).values):
    if c == 0:
        ax_bot.text(i, ax_bot.get_ylim()[1] * 0.05, "n/a\n(clamped)",
                    ha="center", va="bottom", fontsize=6.5, color="gray")
ax_bot.set_ylabel("force-switch PTS gap (ms)")
ax_bot.set_xlabel("Filter delay (s)")
ax_bot.set_title("(b) PTS gap on Ready cells", loc="left", fontsize=8, pad=2)

# Cache boundary marker on both panels.
boundary_x = np.interp(CACHE_SIZE, counts["delay_s"], x)
for a in (ax_top, ax_bot):
    a.axvline(boundary_x, color="black", linestyle="--", linewidth=0.8, alpha=0.6)
    a.set_xticks(x)
    a.set_xticklabels(counts["delay_s"].astype(str))

ax_top.text(boundary_x, 105, f"cache_size = {CACHE_SIZE}",
            fontsize=6.5, ha="center", va="bottom")

In [ ]:
fig.savefig(Path.cwd().parent / "figures" / "fig4_e4_cache_boundary.pdf")
fig.savefig(Path.cwd().parent / "figures" / "fig4_e4_cache_boundary.png", dpi=200)